<a href="https://www.kaggle.com/code/rohithiwalml/fine-tuning?scriptVersionId=296769715" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# Install system dependencies for PDF processing
!pip install pymupdf

!pip install -q transformers peft bitsandbytes trl accelerate datasets

In [ ]:
import subprocess
import re
import json
import sys
import os
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, BitsAndBytesConfig)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer

## Data Cleaning for the pdf 
This cleaning section has the code for research paper, for removing headers, citations, and other refrences !!

In [ ]:
import fitz
import re
def strip_benchmark_scaffold(text):
    # Remove PROMPT / headings
    text = re.sub(r'\bPROMPT\b\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'##\s*(question|choices|instruction)\s*:?', '', text, flags=re.IGNORECASE)
    text = re.sub(r'(evaluation|parse|judge if|ground truth|test cases|stdin|stdout).*','',text,flags=re.IGNORECASE | re.DOTALL)

    # Remove JSON answer schema blocks
    text = re.sub(
        r'\{[^{}]*"answer"[^{}]*\}',
        '',
        text,
        flags=re.DOTALL
    )

    # Remove directive-style prompts
    text = re.sub(
        r'please\s+(answer|write|solve|output).*',
        '',
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    # Remove think tags
    text = re.sub(r'</think>.*', '', text, flags=re.IGNORECASE | re.DOTALL)

    # Final cleanup
    text = re.sub(r'\s+', ' ', text).strip()
    return text
def is_reference_entry(text):
    # Typical reference pattern: names + year + venue
    return bool(re.search(r'\b(19|20)\d{2}\b', text)) and ',' in text and 'Proceedings' in text


def strip_references_section(text):
    # Kill explicit references headers
    if re.match(r'^(references|bibliography)$', text.strip(), re.IGNORECASE):
        return ""
    return text


def normalize_table_references(text):
    # Remove dangling table / figure mentions
    text = re.sub(r'In Table \d+[^,]*,?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'In Figure \d+[^,]*,?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Our results in Table \d+ show that\s*', 'Our results show that ', text)
    return text


def strip_academic_filler(text):
    fillers = [
        r'\bwe observe that\b',
        r'\bas expected\b',
        r'\bit can be seen that\b',
        r'\binterestingly\b',
        r'\bin this work\b',
        r'\bwe find that\b'
    ]
    for f in fillers:
        text = re.sub(f, '', text, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', text).strip()


def high_information_density(text):
    stopwords = len(re.findall(r'\b(the|and|of|to|in|we|that|is)\b', text.lower()))
    words = len(text.split())
    if words == 0:
        return False
    return (stopwords / words) < 0.45

def is_author_contribution(text):
    patterns = [
        r'equal contribution',
        r'listing order is random',
        r'work performed while at',
        r'proposed.*designed.*implemented',
        r'was responsible for'
    ]
    text_l = text.lower()
    return any(re.search(p, text_l) for p in patterns)
    
def is_dataset_meta(text):
    triggers = [
        "conversation between user and assistant",
        "assistant first thinks",
        "reasoning process is enclosed",
        "thinking process",
        "chain-of-thought prompting"
    ]
    text_l = text.lower()
    return any(t in text_l for t in triggers)

def is_exam_question(text):
    return bool(re.search(
        r'which of the following|\(A\)|\(B\)|\(C\)|\(D\)',
        text,
        flags=re.IGNORECASE
    ))

def is_likely_header_footer_or_caption(text):
    t = text.lower().strip()

    if re.match(r'^(figure|table|listing)\s+\d+', t):
        return True
    if "arxiv" in t or "preprint" in t:
        return True
    if len(t) < 5 and t.isdigit():
        return True
    if "@" in t and "." in t:
        return True

    return False
def is_heading(text):
    t = text.strip()

    if t.isupper() and len(t.split()) < 10:
        return True
    if re.match(r'^\d+(\.\d+)*\s+[A-Z]', t):
        return True
    if t.lower() in {
        "abstract", "introduction", "related work", "method",
        "experiments", "results", "discussion", "conclusion",
        "references", "appendix"
    }:
        return True

    return False
def remove_citations(text):
    # (Author et al., 2021)
    text = re.sub(r'\([A-Za-z\s]+et al\.,?\s*\d{4}\)', '', text)
    # [12], [3,4]
    text = re.sub(r'\[[0-9,\s]+\]', '', text)
    return re.sub(r'\s+', ' ', text).strip()
def is_equation(text):
    return bool(re.search(r'[=<>±∑∫√^_]', text))
def is_table_rows(text):
    digits = sum(c.isdigit() for c in text)
    return digits / max(len(text), 1) > 0.4
def is_diagram_label(text):
    return len(text.split()) < 5 and not text.endswith(('.', '!', '?'))
def is_semantically_useful(text):
    # must end properly
    if not text.endswith(('.', '!', '?')):
        return False

    # no broken brackets
    stack = []
    pairs = {'(': ')', '[': ']', '{': '}'}
    for c in text:
        if c in pairs:
            stack.append(c)
        elif c in pairs.values():
            if not stack or pairs[stack.pop()] != c:
                return False
    return not stack

def extract_and_clean_text(pdf_path):
    data = []
    doc = fitz.open(pdf_path)

    for page in doc:
        page_height = page.rect.height
        blocks = page.get_text("blocks")

        for block in blocks:
            if block[6] != 0:  
                continue

            x0, y0, x1, y1, text, block_no, block_type = block

            # Header / footer position filter
            if y1 < 50 or y0 > page_height - 50:
                continue

            if is_likely_header_footer_or_caption(text):
                continue

            if is_heading(text):
                continue

            # Cleaning
            text = re.sub(r'-\n', '', text)
            text = re.sub(r'\n', ' ', text)
            text = re.sub(r'\s+', ' ', text).strip()

            text = remove_citations(text)
            text = strip_benchmark_scaffold(text)
            text = strip_references_section(text)
            text = normalize_table_references(text)
            text = strip_academic_filler(text)
            
            if not text:
                continue
            if is_reference_entry(text):
                continue
            if is_author_contribution(text):
                continue
            if is_dataset_meta(text):
                continue
            if is_exam_question(text):
                continue
            if not high_information_density(text):
                continue
            if text[0].islower() and not text.endswith(('.', '!', '?')):
                continue
            if is_equation(text) or is_table_rows(text) or is_diagram_label(text):
                continue
            if len(text.split()) < 30:
                continue
            if not is_semantically_useful(text):
                continue

            data.append({"text": text})

    return data

In [ ]:
pdf_paths = ['/kaggle/input/the-silentpatient-pdf/The Silent Patient-10-265.pdf']
pdf_pathss = ['/kaggle/input/rlhf-paper2002/rlhf.pdf',
             '/kaggle/input/paper-seek/deepseek12948v2.pdf',
             '/kaggle/input/attention/attention.pdf']
            


all_sft_data = []

for pdf_path in pdf_paths:
    try:
        cleaned = extract_and_clean_text(pdf_path)
        all_sft_data.extend(cleaned)
        print(f"Processed {pdf_path} → {len(cleaned)} blocks")
    except Exception as e:
        print(f"Failed on {pdf_path}: {e}")


In [ ]:
all_sft_data

In [ ]:
data = [p for p in sft_data]

# LoRA

In [ ]:
# For cleaning the GPU VRAM
import torch
import gc

for obj in list(globals().keys()):
    if obj.startswith(("model", "tokenizer", "pipe")):
        del globals()[obj]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("GPU memory cleanup done")


In [ ]:
import shutil
import os

path = "/kaggle/working/tinyllama-lora"

if os.path.exists(path):
    shutil.rmtree(path)
    print("tinyllama-lora removed")
else:
    print("Path does not exist")


In [ ]:
# Path for Base model
model_base = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_base)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

In [ ]:
def tokenize_fn(examples):
    tokens = tokenizer(examples["text"],truncation=True,padding="max_length",max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [ ]:
dataset = Dataset.from_list(all_sft_data)

In [ ]:
tokenized = dataset.map(tokenize_fn, batched = True, remove_columns = ['text'])

In [ ]:
tokenized
split = tokenized.train_test_split(
    test_size=32,
    seed=42
)

test_dataset_silent = split["test"]
train_dataset_silent = split["train"]


In [ ]:
# Downloading 8 bit params.
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,   
    bnb_8bit_compute_dtype=torch.float16
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_base,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

In [ ]:
# LoRA Configuration
lora_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

In [ ]:
non_inst_model_lora = get_peft_model(model, lora_config)

In [ ]:
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=15,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=non_inst_model_lora,
    args=args,
    train_dataset=test_dataset_silent
)

In [ ]:
trainer.train()

In [ ]:
lora_adapter = '/kaggle/working/tinyllama-lora/checkpoint-60'

In [ ]:
prompt = "Who is Alicia Berenson’s husband, and what is his profession?"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
prompt = "Who is Alicia Berenson’s husband, and what is his profession?"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


"""
The model is hallucinating here because model has no context of the story,
model's output is completely vague, the output is just generating questions instead of giving the answe of the asked question.
"""

In [ ]:
model_lora = AutoModelForCausalLM.from_pretrained(
    model_base,
    dtype=torch.float16,
    device_map="auto"
)

model_lora = PeftModel.from_pretrained(model_lora, lora_adapter)

model_lora.eval()

inputs = tokenizer(prompt, return_tensors="pt").to(model_lora.device)

with torch.no_grad():
    output = model_lora.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

"""
Here in this output the Model generated better result compared to base model, model knows what question is asked to it.
it clearly mentioned about her husband's job a famous photgrapher and it also mentioned about he was stalking a woman... 
which is the plot of this story
""" 

## Instruction based SFT

In [ ]:
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")

In [ ]:
dataset
dataset[0]
dataset.column_names

In [ ]:
def format_row(example):
    return {
        "prompt": f"[INST] {example['Context']} [/INST]",
        "response": example["Response"]
    }

dataset = dataset.map(format_row, remove_columns=dataset.column_names)


In [ ]:
MAX_LEN = 512

def tokenize(example):
    prompt_ids = tokenizer(
        example["prompt"],
        truncation=True,
        max_length=MAX_LEN // 2,
        add_special_tokens=False,
    )["input_ids"]

    response_ids = tokenizer(
        example["response"],
        truncation=True,
        max_length=MAX_LEN // 2,
        add_special_tokens=False,
    )["input_ids"]

    input_ids = prompt_ids + response_ids + [tokenizer.eos_token_id]
    labels = [-100]*len(prompt_ids) + response_ids + [tokenizer.eos_token_id]

    return {
        "input_ids": input_ids[:MAX_LEN],
        "labels": labels[:MAX_LEN],
    }
tokenized_dataset = dataset.map(
    tokenize,
    remove_columns=dataset.column_names
)

In [ ]:
split = tokenized_dataset.train_test_split(test_size=0.40, seed=15)

train_dataset = split["train"]
eval_dataset = split["test"]


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

     

In [ ]:
non_instructional_trained_model = AutoModelForCausalLM.from_pretrained(
    model_base,
    quantization_config = bnb_config,
    dtype=torch.float16,
    device_map="auto"
)

In [ ]:
non_instructional_trained_model = get_peft_model(non_instructional_trained_model, lora_config)

In [ ]:
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=non_instructional_trained_model,
    args=args,
    train_dataset=train_dataset
)

In [ ]:
trainer.train()

In [ ]:
sft_model = '/kaggle/working/tinyllama-instruction/checkpoint-792'

In [ ]:
model_sft = PeftModel.from_pretrained(
    model, sft_model)


In [ ]:
model_lora = PeftModel.from_pretrained(model, sft_model)

model_lora.eval()
prompt = "Who is Alicia Berenson’s husband, and what is his profession?"
inputs = tokenizer(prompt, return_tensors="pt").to(model_lora.device)

with torch.no_grad():
    output = model_lora.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


'''Here after instructions SFT the model is generating much better result model output telling about their relationship  also not just about profession'''